
# Phase 3C.03 — Cross-Asset & Lookback Robustness

**Objective:** Evaluate the robustness of Time-Series Momentum across 16 ETFs, four asset classes, and predefined lookbacks (3, 6, 12 months).

The 12-month lookback remains the primary specification. Analysis is restricted to Train (2010–2017) and Validation (2018–2021). Test data remain excluded.


## 1. Environment and Data Preparation

### 1.1. Imports and Configuration

Configure the Python environment and import reusable functions for data preparation, signal construction, and validation.


In [122]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import importlib
import src.data
import src.signals

importlib.reload(src.data)
importlib.reload(src.signals)

from src.data import download_prices, to_monthly_prices
from src.signals import calculate_momentum, generate_signals, clean_signals_and_targets, generate_targets
from src.data import validate_monthly_prices


### 1.2. Data Loading and Integrity Validation

Download and resample adjusted prices for the 16 ETFs over 2008–2021.

Validate data integrity using the reusable `validate_monthly_prices()` function, checking asset coverage, missing values, duplicate dates, chronological order, and monthly continuity.

Invalid datasets must stop the research pipeline.


In [123]:
asset_universe = {
    "Equities" : ["SPY", "VGK", "EWJ", "EEM"],
    "Commodities" : ["USO", "DBA", "DBB", "DBC"],
    "Bonds" : ["IEF", "TLT", "LQD", "HYG"],
    "Forex" : ["FXE", "FXY", "FXB", "FXA"]
}

tickers = []

for assets in asset_universe.values():
    tickers.extend(assets)

prices = download_prices(tickers, "2008-01-01", "2022-01-01", auto_adjust=True, progress=False)

monthly_prices = to_monthly_prices(prices)

validate_monthly_prices(monthly_prices, tickers, "2008-01-31", "2021-12-31")
print("Data validation passed")

Data validation passed



### 1.3. Signal and Target Preparation

Construct time-series momentum signals for the 16 ETFs using **3-, 6-, and 12-month lookback windows**. The 12-month lookback remains the primary specification, while 3 and 6 months are used for robustness analysis. All signals predict the direction of the **next-month return** (1-month prediction horizon).

Split signals and targets according to signal formation dates:
- **Train:** December 2009 – November 2017 (96 months).
- **Validation:** December 2017 – November 2021 (48 months).
- **Test:** Excluded from this phase.

For each lookback, apply `clean_signals_and_targets()` to identify valid signal-target pairs, excluding missing and neutral observations. The primary 12-month specification retains **1,531 Train** and **765 Validation** observations.

For lookback robustness comparisons, construct a **common validity mask** requiring each asset-month observation to be valid across all three lookbacks. Apply this mask to signals and targets using `.where()` to ensure identical evaluation samples.

**Final common sample:**
- **Train:** 1,529 valid observations.
- **Validation:** 762 valid observations.

The cross-asset analysis uses the individual 12-month validity mask, while lookback comparisons use the common sample to eliminate differences caused by sample composition.


In [124]:
momentum_12m = calculate_momentum(monthly_prices, lookback=12)
signals_12m = generate_signals(momentum_12m)

momentum_6m = calculate_momentum(monthly_prices, lookback=6)
signals_6m = generate_signals(momentum_6m)

momentum_3m = calculate_momentum(monthly_prices, lookback=3)
signals_3m = generate_signals(momentum_3m)

targets_1m = generate_targets(monthly_prices, horizon=1)

signals_by_lookback = {
    3: signals_3m,
    6: signals_6m,
    12: signals_12m
}

train_signals_by_lookback = {}
validation_signals_by_lookback = {}

for lookback, signals in signals_by_lookback.items():
    train_lookback = signals.loc["2009-12-31":"2017-11-30"]
    validation_lookback = signals.loc["2017-12-31":"2021-11-30"]

    train_signals_by_lookback[lookback] = train_lookback
    validation_signals_by_lookback[lookback] = validation_lookback

train_targets = targets_1m.loc["2009-12-31":"2017-11-30"]
validation_targets = targets_1m.loc["2017-12-31":"2021-11-30"]

train_mask = {}
validation_mask = {}

for lookback, signals in train_signals_by_lookback.items():
    _, _, mask = clean_signals_and_targets(signals, train_targets)

    train_mask[lookback] = mask

    print(f"Train {lookback}M: {signals.shape}")
    print(f"Train {lookback}M mask: {train_mask[lookback].shape}")
    print(f"Valid observations ({lookback}M): {mask.sum().sum()}")
    print("")


print("\n")

for lookback, signals in validation_signals_by_lookback.items():
    _, _, mask = clean_signals_and_targets(signals, validation_targets)

    validation_mask[lookback] = mask

    print(f"Validation {lookback}M: {signals.shape}")
    print(f"Validation {lookback}M mask: {validation_mask[lookback].shape}")
    print(f"Valid observations ({lookback}M): {mask.sum().sum()}")
    print("")

print("\n")

print("Train targets:", train_targets.shape)
print("Validation targets:", validation_targets.shape)

train_common_mask = train_mask[3] & train_mask[6] & train_mask[12]
validation_common_mask = validation_mask[3] & validation_mask[6] & validation_mask[12]

print("\n")

print(f"Train common mask : {train_common_mask.sum().sum()}")
print(f"Validation common mask : {validation_common_mask.sum().sum()}")

train_signals_common = {}
validation_signals_common = {}

print("\n")

for lookback, signals in train_signals_by_lookback.items():
    train_signals_common[lookback] = train_signals_by_lookback[lookback].where(train_common_mask)
    print(f"Train {lookback}M common signals: {train_signals_common[lookback].shape}")

print("")

for lookback, signals in validation_signals_by_lookback.items():
    validation_signals_common[lookback] = validation_signals_by_lookback[lookback].where(validation_common_mask)
    print(f"Validation {lookback}M common signals: {validation_signals_common[lookback].shape}")

print("\n")

train_targets_common = train_targets.where(train_common_mask)
validation_targets_common = validation_targets.where(validation_common_mask)

for lookback, signals in train_signals_common.items():
    print(f"Train {lookback}M: {signals.notna().sum().sum()}")

print("")

for lookback, signals in validation_signals_common.items():
    print(f"Validation {lookback}M: {signals.notna().sum().sum()}")

print("\n")

print(train_targets_common.notna().sum().sum())
print(validation_targets_common.notna().sum().sum())

Train 3M: (96, 16)
Train 3M mask: (96, 16)
Valid observations (3M): 1530

Train 6M: (96, 16)
Train 6M mask: (96, 16)
Valid observations (6M): 1532

Train 12M: (96, 16)
Train 12M mask: (96, 16)
Valid observations (12M): 1531



Validation 3M: (48, 16)
Validation 3M mask: (48, 16)
Valid observations (3M): 763

Validation 6M: (48, 16)
Validation 6M mask: (48, 16)
Valid observations (6M): 764

Validation 12M: (48, 16)
Validation 12M mask: (48, 16)
Valid observations (12M): 765



Train targets: (96, 16)
Validation targets: (48, 16)


Train common mask : 1529
Validation common mask : 762


Train 3M common signals: (96, 16)
Train 6M common signals: (96, 16)
Train 12M common signals: (96, 16)

Validation 3M common signals: (48, 16)
Validation 6M common signals: (48, 16)
Validation 12M common signals: (48, 16)


Train 3M: 1529
Train 6M: 1529
Train 12M: 1529

Validation 3M: 762
Validation 6M: 762
Validation 12M: 762


1529
762



### Reusable Research Pipeline

The data preparation workflow implemented above has been consolidated into `prepare_momentum_samples()` in `src/research.py`.

This function automates momentum signal construction, target generation, temporal filtering, individual validity masks, and common-sample preparation.

Its outputs were validated against the manual implementation for both Train and Validation. The function will be reused in subsequent research notebooks to ensure consistency and reproducibility.



## 2. Cross-Asset Predictive Analysis

Evaluate the predictive performance of the primary 12-month momentum signal for each of the 16 ETFs.

Compare Train and Validation results to identify cross-asset dispersion and potential instability in predictive accuracy.

### 2.1. Individual Asset Accuracy

Calculate directional prediction accuracy for each ETF using the primary 12-month momentum signal and its individual validity mask. Exclude missing and neutral observations.

In [125]:
train_signals_12m = train_signals_by_lookback[12].where(train_mask[12])
train_targets_12m = train_targets.where(train_mask[12])

print(train_signals_12m.notna().sum().sum())
print(train_targets_12m.notna().sum().sum())

validation_signals_12m = validation_signals_by_lookback[12].where(validation_mask[12])
validation_targets_12m = validation_targets.where(validation_mask[12])

print(validation_signals_12m.notna().sum().sum())
print(validation_targets_12m.notna().sum().sum())

train_comparaison = train_signals_12m == train_targets_12m
train_correct_predictions_by_etf = train_comparaison.sum()
train_valid_observations_by_etf = train_targets_12m.notna().sum()
train_individual_accuracy_by_etf = train_correct_predictions_by_etf / train_valid_observations_by_etf

train_incorrect_predictions_by_etf = train_valid_observations_by_etf - train_correct_predictions_by_etf

# for etf, value in train_individual_accuracy_by_etf.items():
#     print(f"{etf}'s accuracy : {value:.2%}")

# print("\n")

validation_comparaison = validation_signals_12m == validation_targets_12m
validation_correct_predictions_by_etf = validation_comparaison.sum()
validation_valid_observation_by_etf = validation_targets_12m.notna().sum()
validation_individual_accuracy_by_etf = validation_correct_predictions_by_etf / validation_valid_observation_by_etf

validation_incorrect_predictions_by_etf = validation_valid_observation_by_etf - validation_correct_predictions_by_etf

# for etf, value in validation_individual_accuracy_by_etf.items():
#     print(f"{etf}'s accuracy : {value:.2%}")

accuracy_change = (validation_individual_accuracy_by_etf - train_individual_accuracy_by_etf) * 100

individual_accuracy_comparison_df = pd.DataFrame({
    "Train Accuracy" : train_individual_accuracy_by_etf,
    "Validation Accuracy" : validation_individual_accuracy_by_etf,
    "Accuracy Change (PP)" : accuracy_change
})

individual_accuracy_comparison_prct = individual_accuracy_comparison_df.style.format({
    "Train Accuracy" : "{:.2%}",
    "Validation Accuracy" : "{:.2%}",
    "Accuracy Change (PP)" : "{:.2f} pp"
})

# print("\n")

display(individual_accuracy_comparison_prct)

train_mean_accuracy = individual_accuracy_comparison_df["Train Accuracy"].mean()
train_std_accuracy = individual_accuracy_comparison_df["Train Accuracy"].std()

validation_mean_accuracy = individual_accuracy_comparison_df["Validation Accuracy"].mean()
validation_std_accuracy = individual_accuracy_comparison_df["Validation Accuracy"].std()

print(f"Train mean accuracy : {train_mean_accuracy:.2%}")
print(f"Train accuracy standard deviation : {train_std_accuracy * 100:.2f} pp")
print("")
print(f"Validation mean accuracy : {validation_mean_accuracy:.2%}")
print(f"Vaidation standard deviation : {validation_std_accuracy * 100:.2f} pp")

print("")
print("-" * 100)
print("")

positive_accuracy_change = (individual_accuracy_comparison_df["Accuracy Change (PP)"] > 0 ).sum()
negative_accuracy_change = (individual_accuracy_comparison_df["Accuracy Change (PP)"] < 0).sum()
no_accuracy_change = (individual_accuracy_comparison_df["Accuracy Change (PP)"] == 0).sum()

mean_absolute_accuracy_change = individual_accuracy_comparison_df["Accuracy Change (PP)"].abs().mean()

print(f"Positive accuracy change : {positive_accuracy_change}")
print(f"Negative accuracy change : {negative_accuracy_change}")
print(f"No accuracy change : {no_accuracy_change}")
print("")
print(f"Mean absolute accuracy change : {mean_absolute_accuracy_change:.2f} pp")



1531
1531
765
765


,Train Accuracy,Validation Accuracy,Accuracy Change (PP)
Ticker,,,
DBA,48.94%,63.83%,14.89 pp
DBB,46.88%,45.83%,-1.04 pp
DBC,52.08%,54.17%,2.08 pp
EEM,51.04%,47.92%,-3.12 pp
EWJ,52.63%,41.67%,-10.96 pp
FXA,45.83%,41.67%,-4.17 pp
FXB,39.58%,42.55%,2.97 pp
FXE,46.88%,60.42%,13.54 pp
FXY,59.38%,50.00%,-9.38 pp


Train mean accuracy : 51.41%
Train accuracy standard deviation : 8.26 pp

Validation mean accuracy : 51.12%
Vaidation standard deviation : 7.77 pp

----------------------------------------------------------------------------------------------------

Positive accuracy change : 7
Negative accuracy change : 8
No accuracy change : 1

Mean absolute accuracy change : 6.12 pp



### 2.1. Individual Asset Accuracy CCL 

Evaluate the 12-month momentum signal's directional accuracy for each of the 16 ETFs across Train and Validation, excluding neutral and missing observations.

Analyze cross-asset dispersion using mean accuracy and standard deviation, and assess temporal stability through individual accuracy changes and mean absolute accuracy change.

**Key Findings:**
- Mean Accuracy: 51.41% (Train) vs. 51.12% (Validation).
- Cross-asset Standard Deviation: 8.26 pp (Train) vs. 7.77 pp (Validation).
- Accuracy improved for 7 ETFs, declined for 8, and remained unchanged for 1.
- Mean Absolute Accuracy Change: 6.12 pp.

**Interpretation:** Aggregate accuracy remained relatively stable, but individual ETF performance exhibited meaningful variation across periods. These descriptive results do not establish predictive skill, as asset-specific directional baselines and statistical uncertainty have not yet been evaluated.



### 2.2. Individual Asset Baseline Comparison

Compare the 12-month momentum signal's predictive accuracy against Always Long (+1) and Always Short (-1) baselines for each ETF across Train and Validation.

Evaluate all predictions on identical valid observations and calculate the signal's excess accuracy relative to each baseline.

The objective is to determine whether momentum provides additional directional information beyond the underlying distribution of positive and negative returns.


In [130]:
train_always_long_observations = (train_targets_12m == 1).sum()
train_always_long_accuracy = train_always_long_observations / train_valid_observations_by_etf

print("Train always long accuracy by ETF :")
for etf, value in train_always_long_accuracy.items():
    print(f"{etf} : {value:.2%}")

print("")

train_always_short_observations = (train_targets_12m == -1).sum()
train_always_short_accuracy = train_always_short_observations / train_valid_observations_by_etf

print("Train always short accuracy by ETF :")
for etf, value in train_always_short_accuracy.items():
    print(f"{etf} : {value:.2%}")

# print((train_always_long_accuracy + train_always_short_accuracy).round(10)) 

print("\n")

validation_always_long_observations = (validation_targets_12m == 1).sum()
validation_always_long_accuracy = validation_always_long_observations / validation_valid_observation_by_etf

validation_always_short_observations = (validation_targets_12m == -1).sum()
validation_always_short_accuracy = validation_always_short_observations / validation_valid_observation_by_etf

print("Validation always long accuracy by ETF :")
for etf, value in validation_always_long_accuracy.items():
    print(f"{etf} : {value:.2%}")

print("")

print("Validation always short accuracy by ETF :")
for etf, value in validation_always_short_accuracy.items():
    print(f"{etf} : {value:.2%}")

train_excess_accuracy_long = individual_accuracy_comparison_df["Train Accuracy"] - train_always_long_accuracy
train_excess_accuracy_short = individual_accuracy_comparison_df["Train Accuracy"] - train_always_short_accuracy
print("\n")

print("Train excess accuracy long by ETF :")

for etf, value in train_excess_accuracy_long.items():
    print(f"{etf} : {value * 100:.2f}pp")

print("")

print("Train excess accuracy short by ETF :")

for etf, value in train_excess_accuracy_short.items():
    print(f"{etf} : {value * 100:.2f} pp")

validation_excess_accuracy_long = individual_accuracy_comparison_df["Validation Accuracy"] - validation_always_long_accuracy
validation_excess_accuracy_short = individual_accuracy_comparison_df["Validation Accuracy"] - validation_always_short_accuracy

print("\n")

print("Validation excess accuracy long by ETF :")

for etf, value in validation_excess_accuracy_long.items():
    print(f"{etf} : {value * 100:.2f} pp")

print("")

print("Validation excess accuracy short by ETF :")

for etf, value in validation_excess_accuracy_short.items():
    print(f"{etf} : {value * 100:.2f} pp")

train_outperform_both_count = ((train_excess_accuracy_long > 0) & (train_excess_accuracy_short > 0)).sum()
validation_outperform_both_count = ((validation_excess_accuracy_long > 0) & (validation_excess_accuracy_short > 0)).sum()

print("\n")

print(train_outperform_both_count)
print(validation_outperform_both_count)

baseline_comparison_df = pd.DataFrame({
    "Train Momentum" : individual_accuracy_comparison_df["Train Accuracy"],
    "Train Always Long" : train_always_long_accuracy,
    "Train Always Short" : train_always_short_accuracy,
    "Validation Momentum" : individual_accuracy_comparison_df["Validation Accuracy"],
    "Validation Always Long" : validation_always_long_accuracy,
    "Validation Always Short" : validation_always_short_accuracy
    })

baseline_comparison_df_2prct = baseline_comparison_df.style.format({
    "Train Momentum" : "{:.2%}",
    "Train Always Long" : "{:.2%}",
    "Train Always Short" : "{:.2%}",
    "Validation Momentum" : "{:.2%}",
    "Validation Always Long" : "{:.2%}",
    "Validation Always Short" : "{:.2%}"
})

display(baseline_comparison_df_2prct)

Train always long accuracy by ETF :
DBA : 42.55%
DBB : 52.08%
DBC : 45.83%
EEM : 48.96%
EWJ : 63.16%
FXA : 50.00%
FXB : 45.83%
FXE : 51.04%
FXY : 50.00%
HYG : 69.79%
IEF : 52.08%
LQD : 63.54%
SPY : 70.53%
TLT : 51.04%
USO : 47.37%
VGK : 57.29%

Train always short accuracy by ETF :
DBA : 57.45%
DBB : 47.92%
DBC : 54.17%
EEM : 51.04%
EWJ : 36.84%
FXA : 50.00%
FXB : 54.17%
FXE : 48.96%
FXY : 50.00%
HYG : 30.21%
IEF : 47.92%
LQD : 36.46%
SPY : 29.47%
TLT : 48.96%
USO : 52.63%
VGK : 42.71%


Validation always long accuracy by ETF :
DBA : 51.06%
DBB : 54.17%
DBC : 58.33%
EEM : 56.25%
EWJ : 52.08%
FXA : 45.83%
FXB : 53.19%
FXE : 33.33%
FXY : 54.17%
HYG : 62.50%
IEF : 56.25%
LQD : 58.33%
SPY : 70.83%
TLT : 54.17%
USO : 59.57%
VGK : 58.33%

Validation always short accuracy by ETF :
DBA : 48.94%
DBB : 45.83%
DBC : 41.67%
EEM : 43.75%
EWJ : 47.92%
FXA : 54.17%
FXB : 46.81%
FXE : 66.67%
FXY : 45.83%
HYG : 37.50%
IEF : 43.75%
LQD : 41.67%
SPY : 29.17%
TLT : 45.83%
USO : 40.43%
VGK : 41.67%


Train 

,Train Momentum,Train Always Long,Train Always Short,Validation Momentum,Validation Always Long,Validation Always Short
Ticker,,,,,,
DBA,48.94%,42.55%,57.45%,63.83%,51.06%,48.94%
DBB,46.88%,52.08%,47.92%,45.83%,54.17%,45.83%
DBC,52.08%,45.83%,54.17%,54.17%,58.33%,41.67%
EEM,51.04%,48.96%,51.04%,47.92%,56.25%,43.75%
EWJ,52.63%,63.16%,36.84%,41.67%,52.08%,47.92%
FXA,45.83%,50.00%,50.00%,41.67%,45.83%,54.17%
FXB,39.58%,45.83%,54.17%,42.55%,53.19%,46.81%
FXE,46.88%,51.04%,48.96%,60.42%,33.33%,66.67%
FXY,59.38%,50.00%,50.00%,50.00%,54.17%,45.83%



**Key Findings:**

- Only 1 of 16 ETFs outperformed both Always Long and Always Short baselines in Train (FXY).
- Only 1 of 16 ETFs outperformed both baselines in Validation (DBA).
- No ETF outperformed both baselines consistently across Train and Validation.
- SPY achieved 64.58% momentum accuracy in Validation, below the 70.83% Always Long baseline.

**Interpretation:**

The 12-month momentum signal shows limited evidence of consistent directional predictive skill across assets.

High absolute accuracy does not necessarily indicate predictive value when the underlying distribution of returns is directionally imbalanced.

Furthermore, selecting the better constant-direction baseline using realized outcomes is a retrospective benchmark, not an implementable ex-ante trading rule.

These findings do not establish statistical significance or economic profitability.



### 2.3. Cross-Asset Predictive Assessment

Assess the directional predictive performance of the 12-month momentum signal across the 16 ETFs using Balanced Accuracy.

Evaluate whether the signal predicts positive and negative returns effectively, accounting for differences in class distributions.

Compare Train and Validation results to assess cross-asset consistency and identify potential weaknesses in predictive performance.

These results will complement the individual accuracy and baseline comparisons from Sections 2.1 and 2.2.


In [164]:
train_positive_recall_by_etf = ((train_signals_12m == 1) & (train_targets_12m == 1)).sum() / (train_targets_12m == 1).sum()
train_negative_recall_by_etf = ((train_signals_12m == -1) & (train_targets_12m == -1)).sum() / (train_targets_12m == -1).sum()

train_balanced_accuracy_by_etf = (train_positive_recall_by_etf + train_negative_recall_by_etf) / 2

print("Train Balanced accuracy by ETF :")
print(train_balanced_accuracy_by_etf)

validation_positive_recall_by_etf = ((validation_signals_12m == 1) & (validation_targets_12m == 1)).sum() / (validation_targets_12m == 1).sum()
validation_negative_recall_by_etf = ((validation_signals_12m == -1) & (validation_targets_12m == -1)).sum() / (validation_targets_12m == -1).sum()

validation_balanced_accuracy_by_etf = (validation_positive_recall_by_etf + validation_negative_recall_by_etf) / 2

print("\n")

print("Validation balanced accuracy by ETF :")
print(validation_balanced_accuracy_by_etf)

balanced_accuracy_change = (validation_balanced_accuracy_by_etf - train_balanced_accuracy_by_etf) * 100

balanced_accuracy_comparison_df = pd.DataFrame({
    "Train Balanced Accuracy" : train_balanced_accuracy_by_etf,
    "Validation Balanced Accuracy" : validation_balanced_accuracy_by_etf,
    "Balanced Accuracy Change (PP)" : balanced_accuracy_change
})

balanced_accuracy_comparison_df_formatted = balanced_accuracy_comparison_df.style.format({
    "Train Balanced Accuracy" : "{:.2%}",
    "Validation Balanced Accuracy" : "{:.2%}",
    "Balanced Accuracy Change (PP)" : "{:.2f}"
})

display(balanced_accuracy_comparison_df_formatted)

mean_absolute_balanced_accuracy_change = balanced_accuracy_comparison_df["Balanced Accuracy Change (PP)"].abs().mean()
print(f"Mean absolute balanced accuracy : {mean_absolute_balanced_accuracy_change:.2f} pp")

Train Balanced accuracy by ETF :
Ticker
DBA    0.461574
DBB    0.469130
DBC    0.513986
EEM    0.513026
EWJ    0.464286
FXA    0.458333
FXB    0.381119
FXE    0.470473
FXY    0.593750
HYG    0.533968
IEF    0.458696
LQD    0.477283
SPY    0.495469
TLT    0.412940
USO    0.467778
VGK    0.473836
dtype: float64


Validation balanced accuracy by ETF :
Ticker
DBA    0.644022
DBB    0.458042
DBC    0.542857
EEM    0.462963
EWJ    0.408696
FXA    0.412587
FXB    0.421818
FXE    0.578125
FXY    0.496503
HYG    0.433333
IEF    0.510582
LQD    0.535714
SPY    0.455882
TLT    0.428322
USO    0.539474
VGK    0.410714
dtype: float64


,Train Balanced Accuracy,Validation Balanced Accuracy,Balanced Accuracy Change (PP)
Ticker,,,
DBA,46.16%,64.40%,18.24
DBB,46.91%,45.80%,-1.11
DBC,51.40%,54.29%,2.89
EEM,51.30%,46.30%,-5.01
EWJ,46.43%,40.87%,-5.56
FXA,45.83%,41.26%,-4.57
FXB,38.11%,42.18%,4.07
FXE,47.05%,57.81%,10.77
FXY,59.38%,49.65%,-9.72


Mean absolute balanced accuracy : 6.38 pp



**Key Findings:**

- 4 of 16 ETFs achieved Balanced Accuracy above 50% in Train, compared with 6 of 16 in Validation.
- Only DBC maintained Balanced Accuracy above 50% across both periods.
- SPY achieved relatively high conventional Accuracy (68.42% in Train and 64.58% in Validation), but its Balanced Accuracy remained below 50% in both periods.
- Mean Absolute Balanced Accuracy Change: 6.38 pp, compared with 6.12 pp for conventional Accuracy.

**Interpretation:**

The 12-month momentum signal exhibits limited consistency in directional predictive performance across individual assets.

The divergence between conventional Accuracy and Balanced Accuracy highlights the influence of class imbalance, particularly for assets with a high proportion of positive monthly returns.

Although some ETFs achieved Balanced Accuracy above 50%, these results were generally not persistent across Train and Validation.

Observed variations may partly reflect sampling uncertainty. No statistical significance or economic profitability has been established.

**Conclusion:**

The cross-asset analysis provides insufficient evidence of robust, generalized directional predictive skill for the 12-month momentum signal.
